# Incident Risk Classifier
**Lighthouse Sanctuary — ML Pipeline (Lighter)**

Predict whether a resident will experience a **High severity** incident in the next 30 days, enabling proactive preventive intervention.

---
## 1. Problem Framing

### Business Problem
High-severity incidents (medical emergencies, serious behavioral events) are resource-intensive to handle and traumatic for residents. If social workers can identify residents at elevated risk 30 days in advance, they can schedule additional check-ins, counseling sessions, or health evaluations proactively.

### ML Task
- **Type:** Binary classification  
- **Target:** `has_high_incident_next_30d` — did this resident experience any High severity incident in a rolling 30-day window?  
- **Unit:** One record per resident per observation window  
- **Success metric:** High recall on the positive class (minimize missed high-risk residents)

### Features
| Feature | Source |
|---|---|
| `recent_incident_count` | incident_reports (last 90 days) |
| `high_severity_count` | incident_reports (lifetime) |
| `latest_health_score` | health_wellbeing_records |
| `days_in_program` | residents |
| `dominant_start_emotion_enc` | process_recordings (encoded) |
| `progress_noted_rate` | process_recordings |
| `concerns_flagged_rate` | process_recordings |
| `current_risk_level_enc` | residents (encoded) |

### Model
Random Forest (handles class imbalance well, no tuning needed)

### Deployment
No API endpoint. Outputs a risk score table for social worker dashboard.

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, auc, ConfusionMatrixDisplay
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

NB_DIR    = Path().resolve()
ML_DIR    = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR / 'ml-pipelines'
RAW_DIR   = ML_DIR.parent / 'MLPipelines' / 'Data' / 'lighthouse_v7'
DATA_DIR  = ML_DIR / 'data'

RESIDENT_MASTER = DATA_DIR / 'resident_master.csv'
if not RESIDENT_MASTER.exists():
    subprocess.run([sys.executable, str(ML_DIR / 'master_dataset_builder.py')], check=True)

df = pd.read_csv(RESIDENT_MASTER, low_memory=False)

# Also load raw incidents for recency features
inc = pd.read_csv(RAW_DIR / 'incident_reports.csv', low_memory=False)
inc['incident_date'] = pd.to_datetime(inc['incident_date'], errors='coerce')

REFERENCE_DATE = pd.Timestamp('2026-04-07')

print(f'resident_master: {df.shape}')
print(f'incidents: {inc.shape}')

In [ ]:
# Build target: did resident have any High severity incident?
# We'll use as proxy: has_high_incident = high_severity_count > 0
# In a production pipeline with time-ordered data, you'd use a rolling window
df['has_high_incident'] = (df['high_severity_count'].fillna(0) > 0).astype(int)

# Recent incident count (last 90 days from reference date)
cutoff = REFERENCE_DATE - pd.Timedelta(days=90)
recent_inc = inc[inc['incident_date'] >= cutoff].groupby('resident_id').size().reset_index(name='recent_incident_count')
df = df.merge(recent_inc, on='resident_id', how='left')
df['recent_incident_count'] = df['recent_incident_count'].fillna(0)

print(f'High-severity rate: {df["has_high_incident"].mean():.1%}')
print(f'Recent incidents (90d) — per resident:')
print(df['recent_incident_count'].value_counts().head())

In [ ]:
# Encode categorical features
risk_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
df['current_risk_level_enc'] = df['current_risk_level'].map(risk_map).fillna(1)

# Encode dominant start emotion
le = LabelEncoder()
df['emotion_enc'] = le.fit_transform(df['dominant_start_emotion'].fillna('Unknown'))

print('Feature summary:')
feature_cols = [
    'recent_incident_count', 'high_severity_count', 'latest_health_score',
    'days_in_program', 'progress_noted_rate', 'concerns_flagged_rate',
    'current_risk_level_enc', 'emotion_enc'
]
feature_cols = [c for c in feature_cols if c in df.columns]
df[feature_cols].describe()

In [ ]:
# EDA: incident rate by risk level
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

risk_order = ['Low', 'Medium', 'High', 'Critical']
risk_inc_rate = df.groupby('current_risk_level')['has_high_incident'].mean()
risk_inc_rate = risk_inc_rate.reindex([r for r in risk_order if r in risk_inc_rate.index])
axes[0].bar(risk_inc_rate.index, risk_inc_rate.values, color=['#2ecc71','#f39c12','#e67e22','#e74c3c'])
axes[0].set_title('High-Severity Incident Rate by Current Risk Level')
axes[0].set_ylabel('Incident Rate')

# Health score vs incident risk
for label, color in [(0,'#2ecc71'),(1,'#e74c3c')]:
    subset = df[df['has_high_incident'] == label]['latest_health_score'].dropna()
    axes[1].hist(subset, bins=20, alpha=0.6, color=color,
                 label='No High Incident' if label==0 else 'Has High Incident')
axes[1].set_title('Latest Health Score by Incident Status')
axes[1].set_xlabel('Health Score')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 3. Modeling & Feature Selection

In [ ]:
TARGET = 'has_high_incident'

model_df = df[feature_cols + [TARGET]].dropna(subset=[TARGET])
X = model_df[feature_cols].fillna(model_df[feature_cols].median())
y = model_df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Positive rate (train): {y_train.mean():.1%}')

# Random Forest
rf = RandomForestClassifier(
    n_estimators=200, max_depth=6,
    class_weight='balanced',
    random_state=42, n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'\n5-fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

rf.fit(X_train, y_train)
print('Model trained.')

---
## 4. Evaluation & Interpretation

In [ ]:
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_proba)

print(f'Test ROC-AUC: {test_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No High Incident', 'Has High Incident']))

In [ ]:
# ROC curve + confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC={test_auc:.3f}')
axes[0].plot([0,1],[0,1],'k--')
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve — Incident Risk Classifier')
axes[0].legend()

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['No High', 'High'],
    cmap='Reds', ax=axes[1]
)
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
feat_imp.plot(kind='barh', color='#e74c3c')
plt.title('Random Forest Feature Importances — Incident Risk')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print('Top 3 predictors of high-severity incidents:')
for feat, imp in feat_imp.sort_values(ascending=False).head(3).items():
    print(f'  {feat}: {imp:.4f}')

In [ ]:
# Score all active residents
active_df = df[df['case_status'] == 'Active'].copy() if 'case_status' in df.columns else df.copy()
active_X = active_df[feature_cols].fillna(active_df[feature_cols].median())
active_df['incident_risk_score'] = rf.predict_proba(active_X)[:, 1]
active_df['risk_tier'] = pd.cut(
    active_df['incident_risk_score'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low', 'Medium', 'High']
)

print('=== Risk Tier Distribution (Active Residents) ===')
print(active_df['risk_tier'].value_counts().to_string())

print('\n=== Top 10 Highest Risk Residents ===')
print(active_df.nlargest(10, 'incident_risk_score')[
    ['resident_id', 'current_risk_level', 'incident_risk_score',
     'recent_incident_count', 'latest_health_score']
].to_string(index=False))

---
## 5. Causal and Relationship Analysis

In [ ]:
# Does concerns_flagged_rate in counseling sessions predict future incidents?
print('=== Concerns Flagged Rate vs High Incident ===')
concerns_bins = pd.cut(df['concerns_flagged_rate'].fillna(0),
                       bins=[-0.01,0.25,0.5,0.75,1.01],
                       labels=['0-25%','25-50%','50-75%','75-100%'])
concerns_analysis = df.groupby(concerns_bins, observed=True)['has_high_incident'].mean()
print(concerns_analysis.to_string())

plt.figure(figsize=(7, 4))
concerns_analysis.plot(kind='bar', color='#e74c3c')
plt.title('High-Severity Incident Rate vs Concerns Flagged Rate')
plt.ylabel('Incident Rate')
plt.xlabel('Concerns Flagged Rate (% of sessions)')
plt.xticks(rotation=0)
plt.axhline(df['has_high_incident'].mean(), color='black', linestyle='--', label='Overall avg')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Days in program vs incident risk
program_bins = pd.cut(df['days_in_program'].fillna(0),
                       bins=[0, 90, 180, 365, 730, 9999],
                       labels=['<3mo', '3-6mo', '6-12mo', '1-2yr', '2yr+'])
program_analysis = df.groupby(program_bins, observed=True)['has_high_incident'].mean()
print('\n=== Incident Rate by Time in Program ===')
print(program_analysis.to_string())

plt.figure(figsize=(7, 4))
program_analysis.plot(kind='bar', color='#e67e22')
plt.title('High-Severity Incident Rate by Time in Program')
plt.ylabel('Incident Rate')
plt.xlabel('Program Duration')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 6. Deployment Notes

No API endpoint deployed for this model in v1.

### Integration Path
1. Export `incident_risk_score` and `risk_tier` columns into the admin dashboard's Resident List view
2. Sort by `incident_risk_score` descending so social workers see highest-risk residents first
3. Trigger weekly email digest listing "High" tier residents to the safehouse coordinator

### If promoting to production
```python
import joblib
joblib.dump(rf, MODEL_DIR / 'incident_risk_model.pkl')
```
Add endpoint `POST /predict/incident-risk` accepting resident features, returning `{risk_score, risk_tier}`.

### Limitations
- Target variable `has_high_incident` is based on historical data, not future 30-day window (requires time-series split in production)
- Residents with very few records may have unreliable scores
- Retrain every 6 months as resident population changes

### Ethical Considerations
- Incident risk score must never be used punitively against residents
- Treat as a resource allocation tool, not a judgement on the resident